# 08. Hyperparameter tuning (non-SVM) and extra PCA

Parul. SVM is already tuned in `04_classifier_svm.ipynb`. This notebook tunes **logistic regression, random forest, extra trees, and histogram gradient boosting** on the same combined 8206-d train/val split.

PCA is an optional representation (90 / 95 / 99 percent variance), fit on **train only**. Search uses 3-fold CV on train (`f1_macro`). The 541-image val set is scored once for the best config of each model. Results are written under `outputs/tuning/`.


In [ ]:
# Colab setup
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip -q install numpy pandas matplotlib scikit-learn joblib
    if not os.path.exists("/content/MIDS-281-project/.git"):
        !git clone --depth 1 https://github.com/sreeramchennakrish/MIDS-281-project.git /content/MIDS-281-project
    %cd /content/MIDS-281-project/notebooks

print("cwd:", os.getcwd())

## Load features

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold

FEATURE_DIR = Path("../features").resolve()
OUTPUT_DIR = Path("../outputs/tuning").resolve()
MODEL_DIR = Path("../models/tuning").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

with np.load(FEATURE_DIR / "combined_weighted_features.npz") as data:
    X_train = data["X_train"]
    X_val = data["X_val"]
    y_train = data["y_train"]
    y_val = data["y_val"]
    classes = np.asarray(data["classes"])

print("Train:", X_train.shape, "Val:", X_val.shape)
print("Classes:", classes)

## PCA representations (train-only fit)

Keep the full 8206-d vector, plus 90 / 95 / 99 percent variance. If a PCA version does not beat the full vector after tuning, we do not need more PCA for these models.

In [ ]:
reps = {"full 8206-d": (X_train, X_val, X_train.shape[1], 1.0)}
pca_meta = []
for frac in (0.90, 0.95, 0.99):
    pca = PCA(n_components=frac, random_state=42)
    Xtr_p = pca.fit_transform(X_train)
    Xva_p = pca.transform(X_val)
    name = f"PCA {int(frac * 100)}% ({Xtr_p.shape[1]}-d)"
    var = float(pca.explained_variance_ratio_.sum())
    reps[name] = (Xtr_p, Xva_p, Xtr_p.shape[1], var)
    pca_meta.append({"representation": name, "n_components": int(Xtr_p.shape[1]), "variance_kept": var})
    print(name, "variance kept", round(var, 4))

pca_meta_df = pd.DataFrame(pca_meta)
display(pca_meta_df)
pca_meta_df.to_csv(OUTPUT_DIR / "pca_representation_sizes.csv", index=False)

## Search spaces (no SVM)

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

SEARCHES = [
    {
        "name": "logreg",
        # Combined features are already group-standardized in notebook 02. Do not scale again.
        "estimator": LogisticRegression(max_iter=2000, random_state=42),
        "grid": {
            "C": [0.1, 1.0, 10.0],
            "class_weight": [None, "balanced"],
        },
    },
    {
        "name": "random_forest",
        "estimator": RandomForestClassifier(random_state=42, n_jobs=-1),
        "grid": {
            "n_estimators": [200, 400],
            "max_depth": [None, 30],
            "min_samples_leaf": [1, 2],
            "max_features": ["sqrt"],
        },
    },
    {
        "name": "extra_trees",
        "estimator": ExtraTreesClassifier(random_state=42, n_jobs=-1),
        "grid": {
            "n_estimators": [200, 400],
            "max_depth": [None, 30],
            "min_samples_leaf": [1, 2],
            "max_features": ["sqrt"],
        },
    },
    {
        "name": "hist_gb",
        "estimator": HistGradientBoostingClassifier(random_state=42),
        "grid": {
            "learning_rate": [0.05, 0.1],
            "max_depth": [6, None],
            "max_iter": [150],
        },
    },
]


def val_metrics(y_true, y_pred):
    return {
        "val_accuracy": float(accuracy_score(y_true, y_pred)),
        "val_precision_macro": float(precision_score(y_true, y_pred, average="macro")),
        "val_recall_macro": float(recall_score(y_true, y_pred, average="macro")),
        "val_f1_macro": float(f1_score(y_true, y_pred, average="macro")),
    }

## Fit grids and score val once

In [ ]:
rows = []
cv_rows = []
best_bundle = {}

for spec in SEARCHES:
    for rep_name, (Xtr, Xva, n_dim, var_kept) in reps.items():
        # HistGB is slow in 8206-d; only tune it on PCA views.
        if spec["name"] == "hist_gb" and rep_name == "full 8206-d":
            print("skip hist_gb on full 8206-d")
            continue
        print("=" * 60)
        print(spec["name"], "on", rep_name)
        grid = GridSearchCV(
            estimator=spec["estimator"],
            param_grid=spec["grid"],
            scoring="f1_macro",
            cv=cv,
            n_jobs=-1,
            refit=True,
            verbose=1,
        )
        grid.fit(Xtr, y_train)
        pred = grid.predict(Xva)
        metrics = val_metrics(y_val, pred)
        row = {
            "model": spec["name"],
            "representation": rep_name,
            "n_features": n_dim,
            "variance_kept": var_kept,
            "best_cv_f1_macro": float(grid.best_score_),
            "best_params": json.dumps(grid.best_params_),
            **metrics,
        }
        rows.append(row)
        print("best params", grid.best_params_)
        print("CV f1_macro", round(grid.best_score_, 4), "val acc", round(metrics["val_accuracy"], 4))
        print(classification_report(y_val, pred, target_names=classes))
        ConfusionMatrixDisplay.from_predictions(y_val, pred, display_labels=classes, cmap="Blues")
        plt.title(f"{spec['name']} | {rep_name}")
        plt.tight_layout()
        fig_path = OUTPUT_DIR / f"cm_{spec['name']}_{n_dim}d.png"
        plt.savefig(fig_path, dpi=120)
        plt.show()

        cv_df = pd.DataFrame(grid.cv_results_)
        keep = [c for c in cv_df.columns if c.startswith("param_") or c in ("mean_test_score", "std_test_score", "rank_test_score", "mean_fit_time")]
        tmp = cv_df[keep].copy()
        tmp.insert(0, "model", spec["name"])
        tmp.insert(1, "representation", rep_name)
        cv_rows.append(tmp)

        key = f"{spec['name']}__{rep_name}"
        best_bundle[key] = grid.best_estimator_
        joblib.dump(grid.best_estimator_, MODEL_DIR / f"{spec['name']}_{n_dim}d.joblib")

summary = pd.DataFrame(rows).sort_values(["val_accuracy", "best_cv_f1_macro"], ascending=False).reset_index(drop=True)
cv_all = pd.concat(cv_rows, ignore_index=True)

summary.to_csv(OUTPUT_DIR / "val_summary.csv", index=False)
cv_all.to_csv(OUTPUT_DIR / "cv_all_grids.csv", index=False)
with open(OUTPUT_DIR / "val_summary.json", "w") as f:
    json.dump(rows, f, indent=2)

print("Saved tables to", OUTPUT_DIR)
display(summary.round(4))

## Did extra PCA help?

In [ ]:
full = summary[summary["representation"] == "full 8206-d"].set_index("model")["val_accuracy"]
pca_best = (
    summary[summary["representation"] != "full 8206-d"]
    .sort_values("val_accuracy", ascending=False)
    .groupby("model", as_index=False)
    .first()
)
compare = pca_best[["model", "representation", "n_features", "val_accuracy"]].copy()
compare = compare.rename(columns={"val_accuracy": "best_pca_val_acc", "representation": "best_pca_rep"})
compare["full_val_acc"] = compare["model"].map(full)
compare["pca_minus_full"] = compare["best_pca_val_acc"] - compare["full_val_acc"]
compare["pca_worth_using"] = compare["pca_minus_full"] > 0.005
display(compare.round(4))
compare.to_csv(OUTPUT_DIR / "pca_vs_full.csv", index=False)

print("SVM combined val accuracy from notebook 04 (untuned combined): 0.8262")
print("Use pca_worth_using=True only if PCA beats the full vector by more than 0.5 percentage points.")
print("SVM is not retuned here.")